In [4]:
import zipfile
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# STEP 1: LOAD THE DATASET
# ==========================================
path = '/content/AI_Human.csv.zip'
extracted_folder = '/content/dataset/'

print("1. Extracting ZIP file...")

try:
    if not os.path.exists(extracted_folder):
        os.makedirs(extracted_folder)

    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder)
        # Find the extracted CSV file
        csv_files = glob.glob(f"{extracted_folder}/*.csv")
        if not csv_files:
             raise FileNotFoundError("No CSV found inside ZIP.")
        csv_path = csv_files[0]
        print(f"✅ Extracted: {csv_path}")

        # Load CSV using python engine for robustness with large fields
        df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')
        print("✅ Loaded CSV successfully!")

except Exception as e:
    print(f"❌ Extraction Error: {e}")
    # Fallback attempt
    print("Attempting direct read fallback...")
    df = pd.read_csv(path, compression='zip', on_bad_lines='skip')

# Verify columns and handle potential naming variations
print(f"Found columns: {list(df.columns)}")

# Standardize column names
df.columns = [c.lower().strip() for c in df.columns]

# Map common column names
rename_dict = {'text': 'text', 'generated': 'generated', 'essay_text': 'text', 'label': 'generated'}
for old_col, new_col in rename_dict.items():
    if old_col in df.columns and new_col not in df.columns:
        df = df.rename(columns={old_col: new_col})

# ==========================================
# STEP 2: PREPARE DATA
# ==========================================
print("\n2. Preparing Data...")
if 'text' not in df.columns or 'generated' not in df.columns:
    print("❌ ERROR: Could not find required columns. Please check the 'Found columns' list above.")
    display(df.head())
else:
    # Sample data to ensure runtime stability
    if len(df) > 50000:
        df = df.sample(n=50000, random_state=42)

    df = df.dropna(subset=['text', 'generated'])
    X = df['text'].astype(str)
    y = df['generated'].astype(int)

    # ==========================================
    # STEP 3: TRAIN ML MODEL
    # ==========================================
    print("3. Splitting and Vectorizing...")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    print("4. Training Logistic Regression Model...")
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, y_train)

    # ==========================================
    # STEP 4: EVALUATE
    # ==========================================
    print("\n5. Results:")
    y_pred = model.predict(X_test_vec)
    accuracy = accuracy_score(y_test, y_pred)

    print("="*60)
    print(f"⌖ Accuracy: {accuracy * 100:.2f}%")
    print("="*60)
    print(classification_report(y_test, y_pred))

1. Extracting ZIP file...
✅ Extracted: /content/dataset/AI_Human.csv
✅ Loaded CSV successfully!
Found columns: ['text', 'generated']

2. Preparing Data...
3. Splitting and Vectorizing...
4. Training Logistic Regression Model...

5. Results:
⌖ Accuracy: 98.93%
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      6327
           1       0.99      0.98      0.99      3673

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000

